In [15]:
import openai
# from azure.storage.blob import BlobServiceClient
import pickle
# import numpy as np
import os

from azure.search.documents.indexes.models import (  
    SearchIndex,  
    SearchField,  
    SearchFieldDataType,  
    SimpleField,  
    SearchableField,  
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration
)


In [16]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchFieldDataType, SearchableField
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
import re

In [ ]:
import openai

# Azure OpenAI Service client
client = openai.AzureOpenAI(
        azure_endpoint="",
        api_key="",
        api_version="2025-01-01-preview"
    )

model = "text-embedding-ada-002"

# # Generate embeddings using Azure OpenAI Service
# def generate_embeddings(texts):
#     try:
#         response = client.embeddings.create(input=texts, model=model).data
#         embeddings = [item.embedding for item in response]
        
#         # Print the number of embeddings and their shapes
#         print(f"Number of embeddings: {len(embeddings)}")
#         for i, embedding in enumerate(embeddings):
#             print(f"Embedding {i} shape: {len(embedding)}")
        
#         return embeddings
    
#     except Exception as e:
#         print(f"Error generating embeddings: {e}")
#         raise

# 🔹 Generate Embeddings using Azure OpenAI
def generate_embeddings(texts):
    try:
        response = client.embeddings.create(input=texts, model=model).data
        embeddings = [item.embedding for item in response]
        
        print(f"✅ Generated {len(embeddings)} embeddings")
        return embeddings
    
    except Exception as e:
        print(f"❌ Error generating embeddings: {e}")
        raise

In [ ]:
# Azure Cognitive Search details
search_service_name = ""
admin_api_key = ""
schema_index_name = "schema_index" #define new name

# Azure Cognitive Search details
search_service_name = ""
admin_api_key = ""
query_index_name = "query_index" #define new name

# Initialize Search Index client
endpoint = f"https://{search_service_name}.search.windows.net"
credential = AzureKeyCredential(admin_api_key)
index_client = SearchIndexClient(endpoint=endpoint, credential=credential)

In [ ]:

schema_client = SearchClient(
            endpoint=endpoint,
            index_name=schema_index_name,
            credential=credential
        )

# Upload Updated Queries



In [20]:
user_query = "get the those dept which does not have any employee" 

In [21]:

index_name = "schema_index"
def schema_search(query_text, top_k=5):
    try:
        search_client = SearchClient(
            endpoint=endpoint,
            index_name=index_name,
            credential=credential
        )

        from azure.search.documents.models import VectorizedQuery

        # Generate embedding for the query
        query_embedding = generate_embeddings([query_text])[0]  # Ensure it's a single embedding

        vector_query = VectorizedQuery(vector=query_embedding, k_nearest_neighbors=5, fields="content_vector")

        results = search_client.search(search_text="", vector_queries=[vector_query], top=top_k, 
                                        #query_type="semantic",  # Enables AI-driven ranking (optional)
                                        #select=["table_name", "schema_metadata", "@search.score"],  # Select fields
                                        #semantic_configuration_name="my-semantic-config"  # Enable if using semantic search
            )
        
        # print(results)

       

        # Print results
        for result in results:
            print(f"✅ Found: {result['table_name']}")
            print(f"🔹 Schema Metadata: {result['schema_metadata']}")
            print(f"⭐ Score: {result['@search.score']}\n")
        return results
    
    except Exception as e:
        print(f"❌ Error performing search: {e}")
        raise

# # Example Searches
# query_texts = "get all employees"

schema_result  = schema_search(user_query)

for query in schema_result:
    print(f"🔍 Searching for: {query}")
    #schema_search(query)

✅ Generated 1 embeddings
✅ Found: department
🔹 Schema Metadata: 
            Table: department
            Description: Stores department details like department names.
            Columns:
            - dept_id (INT, PK) → Unique identifier for each department.
            - dept_name (VARCHAR) → Name of the department.
        
⭐ Score: 0.8002589

✅ Found: employee
🔹 Schema Metadata: 
            Table: employee
            Description: Stores employee details such as name, salary, and department.
            Columns:
            - emp_id (INT, PK) → Unique identifier for each employee.
            - emp_name (VARCHAR) → Full name of the employee.
            - emp_salary (DECIMAL) → Salary of the employee.
            - dept_id (INT, FK to department) → Foreign key linking to department table.
        
⭐ Score: 0.79291415

✅ Found: leave
🔹 Schema Metadata: 
            Table: leave
            Description: Manages employee leave requests.
            Columns:
            - leave_id 

In [22]:
index_name = "query_index"
def query_search(query_text, top_k=3):
    try:
        search_client = SearchClient(
            endpoint=endpoint,
            index_name=index_name,
            credential=credential
        )
        
        results = search_client.search(query_text, top=top_k)
        return  results #[(result["content"], result, result["@search.score"]) for result in results]
    except Exception as e:
        print(f"Error performing search: {e}")
        raise

# Example usage
# query_text = "get all employees"
query_results = query_search(user_query)

for result in query_results:
    #text, metadata, score = result
    print(f"id: {result["id"]}")
    print(f"query_desc: {result["query_desc"]}")
    print(f"sql_query: {result["sql_query"]}")
    print(f"Score: {result["@search.score"]}\n")


id: query_006
query_desc: Get employees who do not belong to any department
sql_query: 
            SELECT emp_id, emp_name
            FROM employee
            WHERE dept_id IS NULL;
        
Score: 5.1369033

id: query_002
query_desc: Find the highest-paid employee
sql_query: 
            SELECT emp_name, emp_salary 
            FROM employee 
            ORDER BY emp_salary DESC 
            LIMIT 1;
        
Score: 4.4627624

id: query_020
query_desc: Find employees who have not received a promotion
sql_query: 
            SELECT emp_id, emp_name
            FROM employee
            WHERE last_promotion_date IS NULL;
        
Score: 3.1440992



In [23]:
system_prompt = """
You are an expert SQL query generator. Your task is to generate accurate SQL queries based on:  
- A list of predefined queries  
- Schema metadata  

Provided Data:  
Predefined Queries: {predefined_queries}  
Schema Metadata: {schema_result}  

Generate a valid SQL query strictly based on the predefined queries and schema:
"""


In [24]:
system_prompt

'\nYou are an expert SQL query generator. Your task is to generate accurate SQL queries based on:  \n- A list of predefined queries  \n- Schema metadata  \n\nProvided Data:  \nPredefined Queries: {predefined_queries}  \nSchema Metadata: {schema_result}  \n\nGenerate a valid SQL query strictly based on the predefined queries and schema:\n'

In [ ]:
from groq import Groq

client = Groq(api_key="")
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[

       {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ],
    temperature=1,
    max_completion_tokens=1024,
    top_p=1,
    stream=True,
    stop=None,
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")


To generate a SQL query that retrieves the departments without any employees, we can use the following query:

```sql
SELECT d.dept_name 
FROM department d 
LEFT JOIN employee e 
ON d.dept_id = e.dept_id 
WHERE e.dept_id IS NULL;
```

However, since we don't have the actual schema metadata and predefined queries, I'll assume a common schema:

- `department` table with columns `dept_id` and `dept_name`
- `employee` table with columns `emp_id`, `dept_id`, etc.

If the schema is different, the query might need to be adjusted accordingly.

Here's a more general query that should work for most schemas:

```sql
SELECT d.dept_name 
FROM department d 
WHERE d.dept_id NOT IN (SELECT dept_id FROM employee);
```

This query selects all department names from the `department` table where the `dept_id` does not exist in the `employee` table, effectively retrieving departments without any employees.

If you could provide the actual schema metadata and predefined queries, I can generate a more accurat